In [ ]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from get_llm_model import azure_llm_if
llm = azure_llm_if()

from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent

from visualization_system.visualization_backend.analyst.analyst_system import AgenticSystem, PlannerConfig, DirectAnswerConfig
from visualization_system.visualization_backend.analyst.analyst_models import TableItemAgentResponse, SystemPlan, SystemTask 
from visualization_system.visualization_backend.analyst.smart_data import SmartData
from visualization_system.visualization_backend.analyst.smart_data_tools import SmartDataTools
from typing import Any, TypedDict
from pydantic import BaseModel, Field


from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self

class PlannerComponent:
    def __init__(self, llm: Any,  config: PlannerConfig | None = None):
        self.config = config if not config is None else PlannerConfig()
        self.llm = llm

    @property
    def prompt(self) -> str:
        return self.config.prompt

    def run(self, user_query: str,previous_state = None ) -> SystemPlan:
        return self.plan( user_query )
    
    def plan(self, user_query: str, previous_state = None ) -> SystemPlan:
        messages = [
            {"role": "system", "content": self.prompt},
            {"role": "user", "content": user_query},
        ]

        structured_llm = self.llm.with_structured_output(SystemPlan)
        return structured_llm.invoke(messages)
    


class RouterState(BaseModel):
    plan: SystemPlan
    task_index_to_execute: int = 0
    #waiting_for_user: bool = False

class RouterComponent:
    """
    Decide which execution route should run next for the current plan state.

    Note that task_index_to_execute = 0 is the step right after the plan. Plan can be seen as step -1 

    The router does not execute tasks. It only inspects the current `SystemPlan`,
    the current `task_index`, and whether the system is waiting for user input,
    then returns a symbolic route name such as:

    - "direct_answer"
    - "rag_retriever"
    - "data_analysis"
    - "clarification"
    - "aggregate"
    - "end"

    The returned route is later mapped by the graph to the corresponding node.
    """
 
    def route(self, state: RouterState) -> str:
        plan = state.plan

        if plan.clarification_request is not None:
            return "clarification"

        if plan.direct_answer is not None:
            return "direct_answer"

        if state.task_index_to_execute >= len(plan.tasks):
            return "aggregate"

        return plan.tasks[state.task_index_to_execute].agent

    def run(self, state: RouterState) -> str:
        return self.route(state)
